In [17]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [18]:
# Laden dataset Bovemij
df = pd.read_csv('../../data/processed/regression/export_insurance_portfolio_data_forecasting_2015_2025_enriched.csv')

In [19]:
df.head()

,Jaar_Maand,Product,Label_Naam,Beginstand,Jaarpremie_Beginstand,Eindstand,Jaarpremie_Eindstand,NewBusiness,Jaarpremie_NewBusiness,Royementen,...,CPI_Verzekeringen_Vervoer,Jaarmutatie_CPI_Verzekeringen_Vervoer,CPI_Nieuwe_Autos,Jaarmutatie_CPI_Nieuwe_Autos,CPI_Tweedehands_Autos,Jaarmutatie_CPI_Tweedehands_Autos,CPI_Onderhoud_en_Reparatie_Privevoertuigen,Jaarmutatie_CPI_Onderhoud_en_Reparatie_Privevoertuigen,CPI_Brandstoffen,Jaarmutatie_CPI_Brandstoffen
0,201507,Auto Particulier,BCS Polis,50,29585.43,106,59337.54,56,28886.31,0,...,100.09,3.9,99.73,3.2,99.55,1.6,100.25,1.4,105.52,-6.8
1,201508,Auto Particulier,BCS Polis,106,59337.54,168,91685.32,65,32911.85,1,...,100.12,4.2,99.73,2.7,99.88,0.7,100.11,1.0,100.42,-9.7
2,201509,Auto Particulier,BCS Polis,168,91685.32,216,111434.76,54,23342.33,6,...,100.24,3.3,100.27,3.5,100.41,1.6,100.17,1.0,96.83,-13.5
3,201510,Auto Particulier,BCS Polis,216,111434.76,265,133615.83,57,26433.61,6,...,100.21,2.3,100.31,3.4,100.54,1.8,100.27,1.0,95.32,-12.4
4,201511,Auto Particulier,BCS Polis,265,133615.83,309,157925.46,49,26034.01,5,...,100.32,2.3,100.40,3.5,100.98,2.0,100.26,0.6,96.82,-8.6


In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2740 entries, 0 to 2739
Data columns (total 33 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   Jaar_Maand                                              2740 non-null   int64  
 1   Product                                                 2740 non-null   str    
 2   Label_Naam                                              2740 non-null   str    
 3   Beginstand                                              2740 non-null   int64  
 4   Jaarpremie_Beginstand                                   2740 non-null   float64
 5   Eindstand                                               2740 non-null   int64  
 6   Jaarpremie_Eindstand                                    2740 non-null   float64
 7   NewBusiness                                             2740 non-null   int64  
 8   Jaarpremie_NewBusiness                           

In [21]:
# Feature Engineering: Time Features

# Jaar_Maand omzetten naar datetime
df['Datum'] = pd.to_datetime(df['Jaar_Maand'].astype(str), format='%Y%m')

# Extracten van maand en kwartaal uit de datum
df['Year'] = df['Datum'].dt.year
df['Maand'] = df['Datum'].dt.month
df['Kwartaal'] = df['Datum'].dt.quarter

# Binaire feature: Is_Year_Start (1 als het januari is, anders 0), hier vinden veel prolongaties plaats.
df['Is_Year_Start'] = (df['Maand'] == 1).astype(int)

# Cyclische encoding: Zet de maand om in sin_month en cos_month. 
# Dit helpt het model begrijpen dat december (12) en januari (1) dicht bij elkaar liggen.
df['Sin_Maand'] = np.sin(2 * np.pi * df['Maand']/12)
df['Cos_Maand'] = np.cos(2 * np.pi * df['Maand']/12)

In [22]:
df.head()

,Jaar_Maand,Product,Label_Naam,Beginstand,Jaarpremie_Beginstand,Eindstand,Jaarpremie_Eindstand,NewBusiness,Jaarpremie_NewBusiness,Royementen,...,Jaarmutatie_CPI_Onderhoud_en_Reparatie_Privevoertuigen,CPI_Brandstoffen,Jaarmutatie_CPI_Brandstoffen,Datum,Year,Maand,Kwartaal,Is_Year_Start,Sin_Maand,Cos_Maand
0,201507,Auto Particulier,BCS Polis,50,29585.43,106,59337.54,56,28886.31,0,...,1.4,105.52,-6.8,2015-07-01,2015,7,3,0,-0.500000,-8.660254e-01
1,201508,Auto Particulier,BCS Polis,106,59337.54,168,91685.32,65,32911.85,1,...,1.0,100.42,-9.7,2015-08-01,2015,8,3,0,-0.866025,-5.000000e-01
2,201509,Auto Particulier,BCS Polis,168,91685.32,216,111434.76,54,23342.33,6,...,1.0,96.83,-13.5,2015-09-01,2015,9,3,0,-1.000000,-1.836970e-16
3,201510,Auto Particulier,BCS Polis,216,111434.76,265,133615.83,57,26433.61,6,...,1.0,95.32,-12.4,2015-10-01,2015,10,4,0,-0.866025,5.000000e-01
4,201511,Auto Particulier,BCS Polis,265,133615.83,309,157925.46,49,26034.01,5,...,0.6,96.82,-8.6,2015-11-01,2015,11,4,0,-0.500000,8.660254e-01


In [23]:
# Feature Engineering: 

# Define targets and grouping columns
targets = ['Royementen', 'NewBusiness', 'Prolongaties']
group_cols = ['Product', 'Label_Naam']

# 1. Comprehensive Lag Features
# Adding more lags to capture short-term and long-term trends
for target in targets:
    for l in [1, 2, 3, 6, 12]:
        df[f'{target}_lag{l}'] = df.groupby(group_cols)[target].shift(l)


# 2. Rolling Statistics (Volatility & Trends)
for target in targets:
    # 3-month and 6-month moving average
    df[f'{target}_roll_mean6'] = df.groupby(group_cols)[target].transform(lambda x: x.shift(1).rolling(6).mean())
    # Volatility (Standard Deviation) - captures if the market is stable or erratic
    df[f'{target}_roll_std3'] = df.groupby(group_cols)[target].transform(lambda x: x.shift(1).rolling(3).std())


# 3. Momentum & Differences (Velocity of Change)
for target in targets:
    # Absolute change compared to last month
    df[f'{target}_diff1'] = df.groupby(group_cols)[target].diff(1)
    # Percentage change (momentum)
    df[f'{target}_pct_change1'] = df.groupby(group_cols)[target].pct_change(1).replace([np.inf, -np.inf], 0)


De keuze voor deze specifieke getallen ($1, 2, 3, 6, 12$) is niet willekeurig. In de data science en specifiek binnen de verzekeringswereld zijn dit de "standaard bouwstenen" om het gedrag van een tijdreeks te vangen.Hieronder de onderbouwing per onderdeel:


1. De Lag Features: $1, 2, 3, 6, 12$ Lags dwingen het model om naar het verleden te kijken. In plaats van alleen te weten wat er nu gebeurt, geef je het model de context van hoe we hier gekomen zijn.
- Lag 1 & 2 (Het Momentum):
    - Waarom: Verzekeringsdata is vaak "autocorrelated". Als de royementen vorige maand hoog waren, is de kans groot dat ze deze maand ook hoog zijn (bijv. door een aanhoudende ongunstige nieuwsstroom of een prijsverhoging die doorwerkt).
    - Relevantie: Het vangt de directe trend op.
- Lag 3 (Het Kwartaaleffect):
    - Waarom: Veel bedrijfsprocessen en marketingcampagnes werken in cycli van drie maanden.
    - Relevantie: Het helpt bij het identificeren van kortetermijnfluctuaties die langer duren dan één maand, maar korter dan een half jaar.
- Lag 6 (De Halfjaarlijkse Check):
    - Waarom: Dit fungeert als een "ankerpunt" voor de middellange termijn.
    - Relevantie: Het helpt het model te begrijpen of de huidige groei/krimp een tijdelijke uitschieter is of onderdeel van een trend die al een half jaar gaande is.
- Lag 12 (De Seizoensgebondenheid - Cruciaal):
    - Waarom: Verzekeringen zijn jaarproducten. Veel polissen hebben een hoofdvervaldatum die elk jaar op hetzelfde moment valt (vaak 1 januari).
    - Relevantie: Zonder lag 12 snapt een model niet waarom de prolongaties in januari ineens omhoog schieten. Dit is de belangrijkste feature voor je Prolongaties voorspelling.


2. Rolling Statistics
- Rolling Mean 6 (Het Voortschrijdend Gemiddelde)
    - Onderbouwing: Maandcijfers kunnen erg "vlekkerig" zijn (de ene maand 10, de andere 40). Dit noemen we ruis.
    - Relevantie: Door het gemiddelde van de afgelopen 6 maanden te nemen, filter je die ruis weg. Het model krijgt zo een stabiel signaal van de "gezondheid" van de portefeuille. Als de roll_mean6 van Royementen stijgt, is er iets structureel mis, ongeacht of de laatste maand toevallig laag was.
    - Waarom 6 en niet 12? 12 maanden is vaak te traag; tegen de tijd dat het 12-maands gemiddelde reageert, is de markt alweer veranderd. 6 maanden is de "sweet spot" tussen stabiliteit en actualiteit.
- Rolling Std 3 (De Volatiliteit/Onrust)
    - Onderbouwing: De standaarddeviatie meet de spreiding.
    - Relevantie: Een hoge std over 3 maanden betekent dat de instroom of uitstroom heel onvoorspelbaar is (bijv. de ene maand 5, dan 50, dan 10).
    - XGBoost voordeel: XGBoost kan dit gebruiken als een "onzekerheidsfactor". Als de markt onrustig is (hoge std), zal het model minder zwaar leunen op de lag1 en meer kijken naar andere factoren zoals de CPI of Autoverkopen.


3. Momentum & Differences (Velocity of Change)

Deze features richten zich op de dynamiek van de data: niet waar we staan, maar hoe snel we bewegen. In de natuurkunde noemen we dit "snelheid" en "versnelling". Voor een XGBoost-model zijn dit krachtige indicatoren omdat ze helpen bij het detecteren van trendbreuken en marktschokken.

Hier is de onderbouwing voor deze twee specifieke berekeningen:
- target_diff1 (Absolute verandering)
    - Wat het doet: Het berekent het verschil tussen deze maand en de vorige maand (bijv. 120 royementen deze maand minus 100 vorige maand = +20).
    - Waarom relevant:
        - Volume-impact: Voor een bedrijf is het cruciaal om te weten of de werkdruk of de uitstroom in absolute aantallen toeneemt. Een toename van 20 royementen is een concreet signaal.
        - Detectie van "stappen": XGBoost (een boomstructuur-model) kan heel makkelijk drempelwaarden vinden op dit verschil. Bijvoorbeeld: "ALS het aantal royementen met meer dan 15 is gestegen t.o.v. vorige maand, DAN is er waarschijnlijk een structureel probleem."
        - Normalisatie van groei: Het haalt het "niveau" uit de data en kijkt puur naar de mutatie.
- target_pct_change1 (Momentum / Relatieve versnelling)
    - Wat het doet: Het kijkt naar de procentuele groei of krimp (bijv. van 100 naar 120 is +20%).
    - Waarom relevant:
        - Vroegtijdige waarschuwing (Leading Indicator): In een klein label valt een stijging van 5 naar 10 royementen in absolute zin (diff1) niet op, maar het is wel een verdubbeling (+100%). Percentage-verandering legt dit feilloos bloot. Het model leert hierdoor dat er iets gaande is bij dat label, nog voordat de absolute aantallen groot worden.
        - Vergelijking tussen labels: Hiermee kan het model het gedrag van een klein label (100 polissen) vergelijken met een groot label (10.000 polissen) op een "level playing field". Een groei van 10% is voor beide een vergelijkbaar signaal van marktdynamiek.
        - Marktsignalen: Grote procentuele schommelingen wijzen vaak op externe factoren (zoals een nieuwe concurrent of een gewijzigde premie). XGBoost gebruikt dit om te bepalen of de huidige maand een "uitschieter" is of het begin van een nieuwe trend.


Waarom de combinatie van beide?

Stel je voor:
- Scenario A: Royementen gaan van 1000 naar 1010 (+10 diff, +1% pct).
- Scenario B: Royementen gaan van 10 naar 20 (+10 diff, +100% pct).

Zonder deze features ziet het model in beide gevallen een stijging van 10 (diff). Door beide toe te voegen, snapt het model dat Scenario B veel alarmerender is (exponentiële groei) dan Scenario A (minimale fluctuatie in een groot bestand).



In [24]:
df.head()

,Jaar_Maand,Product,Label_Naam,Beginstand,Jaarpremie_Beginstand,Eindstand,Jaarpremie_Eindstand,NewBusiness,Jaarpremie_NewBusiness,Royementen,...,NewBusiness_roll_mean6,NewBusiness_roll_std3,Prolongaties_roll_mean6,Prolongaties_roll_std3,Royementen_diff1,Royementen_pct_change1,NewBusiness_diff1,NewBusiness_pct_change1,Prolongaties_diff1,Prolongaties_pct_change1
0,201507,Auto Particulier,BCS Polis,50,29585.43,106,59337.54,56,28886.31,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,201508,Auto Particulier,BCS Polis,106,59337.54,168,91685.32,65,32911.85,1,...,NaN,NaN,NaN,NaN,1.0,0.000000,9.0,0.160714,0.0,NaN
2,201509,Auto Particulier,BCS Polis,168,91685.32,216,111434.76,54,23342.33,6,...,NaN,NaN,NaN,NaN,5.0,5.000000,-11.0,-0.169231,0.0,NaN
3,201510,Auto Particulier,BCS Polis,216,111434.76,265,133615.83,57,26433.61,6,...,NaN,5.859465,NaN,0.0,0.0,0.000000,3.0,0.055556,0.0,NaN
4,201511,Auto Particulier,BCS Polis,265,133615.83,309,157925.46,49,26034.01,5,...,NaN,5.686241,NaN,0.0,-1.0,-0.166667,-8.0,-0.140351,0.0,NaN


In [25]:
# --- FEATURE ENGINEERING: CPI INTERACTIES ---

# 1. Relatieve Onderhoudskosten vs. Verzekering
# Meet de druk op het variabele budget t.o.v. de vaste verzekeringslasten
df['CPI_Maintenance_vs_Insurance'] = (
    df['Jaarmutatie_CPI_Onderhoud_en_Reparatie_Privevoertuigen'] - 
    df['Jaarmutatie_CPI_Verzekeringen_Vervoer']
)

# 2. De "Occasion" trend t.o.v. Nieuw
# Belangrijk voor de dagwaarde-ontwikkeling van je portefeuille
df['CPI_Used_vs_New_Car_Trend'] = (
    df['Jaarmutatie_CPI_Tweedehands_Autos'] - 
    df['Jaarmutatie_CPI_Nieuwe_Autos']
)

# 3. Totale Auto-inflatie Index (Aggregaat)
# Een 'pijn-index' die laat zien hoe hard het totale autobudget wordt geraakt
df['Total_Auto_Inflation_Index'] = (
    df['Jaarmutatie_CPI_Brandstoffen'] + 
    df['Jaarmutatie_CPI_Onderhoud_en_Reparatie_Privevoertuigen'] + 
    df['Jaarmutatie_CPI_Verzekeringen_Vervoer']
) / 3

# 4. Premie-elasticiteit Proxy
# Voelt de premie 'duur' t.o.v. de aanschafwaarde van een nieuwe auto?
df['Premium_vs_New_Car_Price_Index'] = (
    df['Jaarmutatie_CPI_Verzekeringen_Vervoer'] - 
    df['Jaarmutatie_CPI_Nieuwe_Autos']
)

# --- FEATURE ENGINEERING: VERTRAAGDE EFFECTEN ---

# 5. Vertraagde Brandstofeffecten (Lag 3)
# Consumenten passen hun gedrag vaak pas aan na een paar maanden hoge brandstofprijzen
df['CPI_Brandstoffen_Lag3'] = df.groupby(['Product', 'Label_Naam'])['Jaarmutatie_CPI_Brandstoffen'].shift(3)

# 6. Prijs-Schok Indicator
# Een binaire vlag (0 of 1) voor maanden waarin de verzekeringspremie in de markt extreem stijgt (>5% mutatie-toename)
df['Insurance_Price_Shock'] = (df['Jaarmutatie_CPI_Verzekeringen_Vervoer'].diff() > 5).astype(int)

De laatste set features die we hebben toegevoegd op basis van de CPI-cijfers, vormt de brug tussen de macro-economie (de wereld buiten de verzekeraar) en het individuele gedrag van de klant. In een XGBoost-model helpen deze features om te begrijpen waarom een trend uit het verleden (de lags) plotseling kan veranderen.

Hier is de onderbouwing per feature:

1. CPI_Maintenance_vs_Insurance
- De Logica: Een auto bezitten kost geld aan de 'voorkant' (verzekering/belasting) en aan de 'achterkant' (onderhoud/reparatie). Als de kosten voor onderhoud (onderdelen, uurloon garage) harder stijgen dan de verzekeringspremie, komt het totale autobudget van een huishouden onder druk te staan.
- Relevantie voor Model: Dit is een sterke voorspeller voor Mutaties. Klanten die geconfronteerd worden met dure reparaties, gaan vaak op zoek naar besparingen elders, bijvoorbeeld door hun dekking te verlagen van All-Risk naar WA-Beperkt Casco om de maandlasten te drukken.

2. CPI_Used_vs_New_Car_Trend
- De Logica: Dit meet de spanning op de occasionmarkt. Als tweedehands auto's relatief sneller duurder worden dan nieuwe auto's (een positieve waarde), stijgt de economische waarde van de huidige portefeuille.
- Relevantie voor Model: * NewBusiness: Een oververhitte occasionmarkt zorgt vaak voor meer transacties (mensen verkopen hun auto sneller vanwege de hoge inruilwaarde), wat leidt tot meer nieuwe verzekeringsaanvragen.
- Royementen: Bij een hoge restwaarde zijn mensen minder snel geneigd hun auto weg te doen of de verzekering op te zeggen.

3. Total_Auto_Inflation_Index
- De Logica: Dit is de "pijn-index" voor de automobilist. Het combineert brandstof, onderhoud en verzekering tot één cijfer.
- Relevantie voor Model: XGBoost is uitstekend in het vinden van kantelpunten. Er is vaak een psychologische grens (bijv. 8% totale inflatie) waarbij consumenten massaal hun contracten gaan herzien. Deze geaggregeerde index helpt het model om dit "massa-effect" te herkennen, wat je niet ziet als je alleen naar de losse componenten kijkt.

4. Premium_vs_New_Car_Price_Index
- De Logica: Dit meet de "onrechtvaardigheid" van de premie. Als een nieuwe auto nauwelijks duurder wordt, maar de verzekeringspremie stijgt wel hard, dan voelt de klant dat de verhouding tussen de waarde van het object en de kosten van de verzekering zoek is.
- Relevantie voor Model: Dit is een directe drijver voor Royementen. Het is een maatstaf voor de prijsgevoeligheid; hoe hoger dit getal, hoe groter de kans dat een klant overstapt naar een concurrent die (nog) niet de prijzen heeft verhoogd.

5. CPI_Brandstoffen_Lag3
- De Logica: Mensen reageren niet direct op de brandstofprijs van vandaag. Ze wachten meestal een paar maanden ("is dit een tijdelijke piek?") voordat ze hun gedrag aanpassen (bijv. de tweede auto wegdoen of minder gaan rijden).
- Relevantie voor Model: Dit geeft het model een "vooruitziende blik". Een hoge brandstofprijs nu is vaak een voorspeller voor minder NewBusiness over drie maanden, omdat de verkoop van auto's dan pas begint af te remmen.

6. Insurance_Price_Shock
- De Logica: Dit is een binaire feature (0 of 1). Mensen accepteren kleine, geleidelijke prijsverhogingen vaak wel, maar reageren fel op plotselinge schokken.
- Relevantie voor Model: In een 'Decision Tree' model zoals XGBoost is een 0/1 indicator goud waard. Het model kan hiermee heel simpel een tak splitsen: "Was er deze maand een schok? Ja? Gebruik dan een andere rekenmethode voor Royementen dan in rustige maanden."


Samenvattend: Deze features maken je model empathisch. Ze vertellen het model niet alleen dat de cijfers veranderen, maar geven ook de economische reden waarom een klant besluit op te zeggen, te blijven of een nieuwe polis af te sluiten.

In [26]:
df.head()

,Jaar_Maand,Product,Label_Naam,Beginstand,Jaarpremie_Beginstand,Eindstand,Jaarpremie_Eindstand,NewBusiness,Jaarpremie_NewBusiness,Royementen,...,NewBusiness_diff1,NewBusiness_pct_change1,Prolongaties_diff1,Prolongaties_pct_change1,CPI_Maintenance_vs_Insurance,CPI_Used_vs_New_Car_Trend,Total_Auto_Inflation_Index,Premium_vs_New_Car_Price_Index,CPI_Brandstoffen_Lag3,Insurance_Price_Shock
0,201507,Auto Particulier,BCS Polis,50,29585.43,106,59337.54,56,28886.31,0,...,NaN,NaN,NaN,NaN,-2.5,-1.6,-0.500000,0.7,NaN,0
1,201508,Auto Particulier,BCS Polis,106,59337.54,168,91685.32,65,32911.85,1,...,9.0,0.160714,0.0,NaN,-3.2,-2.0,-1.500000,1.5,NaN,0
2,201509,Auto Particulier,BCS Polis,168,91685.32,216,111434.76,54,23342.33,6,...,-11.0,-0.169231,0.0,NaN,-2.3,-1.9,-3.066667,-0.2,NaN,0
3,201510,Auto Particulier,BCS Polis,216,111434.76,265,133615.83,57,26433.61,6,...,3.0,0.055556,0.0,NaN,-1.3,-1.6,-3.033333,-1.1,-6.8,0
4,201511,Auto Particulier,BCS Polis,265,133615.83,309,157925.46,49,26034.01,5,...,-8.0,-0.140351,0.0,NaN,-1.7,-1.5,-1.900000,-1.2,-9.7,0


In [27]:
# --- FEATURE ENGINEERING: GEAVANCEERDE PREMIE DYNAMIEK ---

# 1. Relatieve Prolongatie Verhoging (%)
# Dit meet de 'pijn' van de verlenging voor de klant.
df['Prolongatie_Pct_Increase'] = (
    df['Jaarpremie_na_Prolongatie'] / 
    df['Jaarpremie_voor_Prolongatie'].replace(0, np.nan)
) - 1

# 2. Premium Gap (Nieuw vs. Bestaand)
# Is er sprake van een 'loyaliteitsboete'? (nieuwe klanten betalen minder dan bestaande)
# We hergebruiken hier de eerder gemaakte 'Avg_Premium_Per_New_Policy'
df['Avg_Premium_Per_New_Policy'] = df['Jaarpremie_NewBusiness'] / df['NewBusiness'].replace(0, np.nan)
df['New_vs_Existing_Premium_Gap'] = (
    df['Avg_Premium_Per_New_Policy'] / 
    df['Gemiddelde_Jaarpremie_Beginstand'].replace(0, np.nan)
)

# 3. Gemiddelde Premie van Vertrokken Klanten
# Verlies je de dure of de goedkope polissen?
df['Avg_Premium_Churned_Policy'] = (
    df['Geroyeerde_Jaarpremie'] / 
    df['Royementen'].replace(0, np.nan)
)

# 4. Gemiddelde Impact van Mutaties
# Gaan klanten bij wijzigingen gemiddeld meer of minder betalen? (Upselling vs. Downselling)
df['Avg_Mutation_Premium_Impact'] = (
    (df['Jaarpremie_na_Mutatie'] - df['Jaarpremie_voor_Mutatie']) / 
    df['Mutaties'].replace(0, np.nan)
)

# 5. Portfolio Yield Trend
# Ontwikkeling van het gemiddelde prijsniveau in de hele portefeuille
df['Portfolio_Yield_Trend'] = (
    df['Gemiddelde_Jaarpremie_Eindstand'] / 
    df['Gemiddelde_Jaarpremie_Beginstand'].replace(0, np.nan)
)

De laatste set features die we hebben toegevoegd richt zich op de interne prijsdynamiek en de winstgevendheid per polis. Waar de CPI-cijfers iets zeggen over de buitenwereld, zeggen deze features alles over hoe jouw specifieke prijsbeleid en de samenstelling van je klantenbestand veranderen.

Hier is de onderbouwing per feature:

1. Prolongatie_Pct_Increase (De 'Prijs-Prikkel')
- De Logica: Een absolute verhoging van €50 op een premie van €500 (10%) voelt voor een klant veel zwaarder dan €50 op een premie van €2000 (2,5%).
- Waarom voor XGBoost: Dit is vaak de sterkste voorspeller voor Royementen. Consumenten hebben een "pijngrens" (vaak rond de 5-10%). Zodra de procentuele verhoging boven die grens komt, stijgt de kans dat ze naar een vergelijker gaan. Door dit als percentage te geven, leert XGBoost precies waar die grens ligt voor jouw labels.

2. New_vs_Existing_Premium_Gap (De 'Loyaliteitsboete')
- De Logica: Dit vergelijkt wat een nieuwe klant gemiddeld betaalt met wat een bestaande klant betaalt.
- Waarom voor XGBoost: Als nieuwe klanten veel minder betalen (gap < 1), is er een sterke prikkel voor bestaande klanten om op te zeggen en ergens anders (of via een omweg bij jou) opnieuw klant te worden. Het verklaart waarom mensen weggaan, zelfs als hun eigen premie maar weinig gestegen is: de markt om hen heen is simpelweg goedkoper.

3. Avg_Premium_Churned_Policy (De 'Kwaliteit' van uitstroom)
- De Logica: Verlies je klanten die weinig premie betalen (vaak WA-verzekerd, oudere auto's) of klanten die veel premie betalen (All-risk, nieuwe auto's)?
- Waarom voor XGBoost: Dit helpt het model om de impact op de Jaarpremie_Eindstand te voorspellen. Als dit getal stijgt, "lekt" er kwalitatief kostbaardere business weg. Het kan ook een signaal zijn dat een specifiek duurder segment (bijv. jonge bestuurders of Tesla-rijders) ontevreden is.

4. Avg_Mutation_Premium_Impact (Downselling vs. Upselling)
- De Logica: Wanneer klanten hun polis wijzigen (Mutaties), gaan ze dan gemiddeld meer of minder betalen?
- Waarom voor XGBoost: Dit is een vroegtijdig waarschuwingssignaal. Als dit getal negatief wordt, zijn klanten hun dekking aan het verlagen (bijv. van All-risk naar WA-beperkt casco) om kosten te sparen. In de data zie je vaak dat klanten die eerst downselling doen, drie maanden later alsnog helemaal opzeggen. Het geeft je model een kijkje in de toekomst.

5. Portfolio_Yield_Trend (Rendementsontwikkeling)
- De Logica: Stijgt of daalt de gemiddelde premie van de totale portefeuille over tijd?
- Waarom voor XGBoost: Dit geeft de algemene "gezondheid" en koers van het label aan. Een stijgende yield is positief voor de omzet, maar als XGBoost ziet dat dit gepaard gaat met een stijgende Royementen_Ratio, leert het model dat het label de grens van wat klanten willen betalen heeft bereikt.


Samenvattend: Wat bereik je hiermee?

Door deze features toe te voegen, dwing je het model om niet alleen naar aantallen te kijken, maar ook naar de onderliggende waarde.
In de verzekeringswereld is de interactie tussen deze factoren goud waard:
- Hoge Prolongatie_Pct_Increase + Grote New_vs_Existing_Premium_Gap = Voorspelling: Explosie in Royementen.

In [28]:
# --- FEATURE ENGINEERING: EXPERT DYNAMICS ---

# 1. New Business Kwaliteit Ratio (Adverse Selection Check)
# Halen we relatief 'duurdere' of 'goedkopere' risico's binnen dan ons huidige gemiddelde?
df['New_Business_Risk_Premium_Ratio'] = (
    df['Avg_Premium_Per_New_Policy'] / 
    df['Gemiddelde_Jaarpremie_Beginstand'].replace(0, np.nan)
)

# 2. Acceleration Features (De versnelling van de trend)
# Is de groei/krimp aan het versnellen of vertragen? (Tweede afgeleide)
for target in ['Royementen', 'NewBusiness', 'Prolongaties']:
    # We pakken de diff van de diff (verandering van de verandering)
    df[f'{target}_acceleration'] = df.groupby(['Product', 'Label_Naam'])[f'{target}_diff1'].diff(1)

# 3. Premium Market Density
# Hoeveel premie-waarde halen we uit de markt per verkochte auto in Nederland?
df['Premium_Market_Density'] = (
    df['Jaarpremie_NewBusiness'] / 
    df['CBS_Autoverkopen_Totaal'].replace(0, np.nan)
)

# 4. Delta Portfolio Waarde (De 'groei-motor')
# De totale jaarpremie-ontwikkeling van de portefeuille
df['Portfolio_Value_Growth_Ratio'] = (
    df['Jaarpremie_Eindstand'] / 
    df['Jaarpremie_Beginstand'].replace(0, np.nan)
)

# 5. Seasonality: Maanden sinds jaarstart
# Helpt het model om 'uitputting' van seizoenseffecten na januari te herkennen
df['Months_Since_Year_Start'] = df['Datum'].dt.month - 1

Deze "Expert Features" tillen je model naar een hoger niveau omdat ze niet alleen kijken naar wat er gebeurt, maar naar de onderliggende dynamiek en de kwaliteit van de veranderingen. Voor een XGBoost-model zijn dit krachtige signalen om complexe marktsituaties te onderscheiden.

Hier is de gedetailleerde onderbouwing per feature:

1. New Business Risk Premium Ratio (Adverse Selection Check)
- De Logica: Vergelijkt de gemiddelde premie van nieuwe klanten met die van de bestaande portefeuille.
- Waarom voor XGBoost: In de verzekeringswereld is premie vaak een proxy voor risico (of dekking).
    - Als dit getal fors daalt, haal je relatief "lichtere" risico's binnen (bijv. meer WA t.o.v. All-Risk).
    - Dit helpt het model te voorspellen dat de toekomstige Jaarpremie_Eindstand minder hard zal groeien dan het aantal polissen doet vermoeden. Het helpt ook bij het detecteren van "adverse selection": trek je de juiste klanten aan tegen de juiste prijs?

2. Acceleration Features (De 'Versnelling')
- De Logica: Dit is de tweede afgeleide (de verandering van de verandering). diff1 is snelheid, acceleration is versnelling.
- Waarom voor XGBoost: Bomen (trees) in XGBoost reageren heel sterk op drempelwaarden.
    - Een stijging in Royementen is één ding, maar een versnellende stijging (hoge acceleratie) duidt op een trendbreuk of een markt-schok (bijv. een concurrent die een agressieve actie start).
    - Het stelt het model in staat om "omslagpunten" te herkennen voordat de absolute aantallen hun piek bereiken.

3. Premium Market Density
- De Logica: De verhouding tussen jouw nieuwe premie-volume en de totale landelijke autoverkopen.
- Waarom voor XGBoost: Dit is een zuivere maatstaf voor commerciële effectiviteit.
    - Als de autoverkopen dalen maar jouw "density" stijgt, dan win je marktaandeel in het waardevolle segment.
    - Het model leert hiermee externe economische krimp (CBS_Autoverkopen) te scheiden van jouw eigen verkoopkracht. Het voorkomt dat het model "pessimistisch" wordt puur omdat de markt krimpt, zolang jij maar een groter stuk van de taart pakt.

4. Portfolio Value Growth Ratio (De 'Groei-motor')
- De Logica: De netto waarde-ontwikkeling van de totale jaarpremie (Eindstand / Beginstand).
- Waarom voor XGBoost: Dit vat alle bewegingen (instroom, uitstroom, premieverhogingen en mutaties) samen in één getal.
    - Het fungeert als een "momentum-indicator" voor de financiële gezondheid van een label.
    - Voor het model is dit een cruciale feature om te bepalen of een label in een "winning loop" zit (gezonde groei) of dat het label aan het "uithollen" is (aantallen blijven gelijk, maar de waarde daalt).

5. Months Since Year Start (Seizoens-uitputting)
- De Logica: Het aantal maanden dat verstreken is sinds januari.
- Waarom voor XGBoost: De verzekeringsmarkt werkt in een jaarlijkse cyclus. Januari is vaak de maand van de grote mutaties en prolongaties. Naarmate het jaar vordert, neemt de "neiging" van consumenten om over te stappen vaak af (inertie).
    - Door dit getal te geven, leert het model dat een stijging van 5% in Royementen in februari (vlak na de piek) heel anders geïnterpreteerd moet worden dan een stijging van 5% in oktober. Het geeft context aan de seizoensgebondenheid die verder gaat dan alleen de naam van de maand.


Samenvatting voor je model-architectuur:

Door deze features toe te voegen, dwing je XGBoost om verder te kijken dan simpele correlaties. Je geeft het model de instrumenten om kwalitatieve verschuivingen (Risk Ratio), markteffectiviteit (Density) en trendversnellingen (Acceleration) te begrijpen. Dit resulteert in een model dat niet alleen kijkt naar het verleden, maar ook begrijpt waarom de toekomst gaat afwijken.

In [29]:
# --- FEATURE ENGINEERING: PORTFOLIO & BENCHMARKS ---

# 1. Retention Rate
# Geeft aan welk deel van de portefeuille behouden blijft (genormaliseerde maatstaf voor churn)
df['Retention_Rate'] = 1 - (df['Royementen'] / df['Beginstand'].replace(0, np.nan))

# 2. Market Penetration Proxy (Market Capture Index)
# Hoe goed presteert de instroom ten opzichte van de totale landelijke autoverkopen?
df['Market_Capture_Index'] = (
    df['NewBusiness'] / 
    df['CBS_Autoverkopen_Totaal'].replace(0, np.nan)
)

# 3. Target Encoding Proxy (Historical Baseline)
# Geeft het model een referentiekader: wat is een 'normaal' aantal royementen voor dit specifieke label?
df['Label_Historical_Avg_Royementen'] = df.groupby('Label_Naam')['Royementen'].transform('mean')

Deze laatste drie features vormen de ruggengraat van je model omdat ze context en normalisatie toevoegen. Zonder deze features zou het model appels met peren vergelijken (bijvoorbeeld een groot label met een klein label).

Hier is de onderbouwing voor deze specifieke "Portfolio & Benchmarks" features:

1. Retention Rate (Klantbehoud)
- De Logica: Dit is de inverse van de churn rate. Het berekent welk percentage van de klanten aan het begin van de maand (Beginstand) aan het einde van de maand nog steeds klant is.
- Waarom voor XGBoost: * Normalisatie: In een dataset met labels van verschillende groottes zijn absolute aantallen royementen misleidend. 100 opzeggingen bij een label van 1.000 polissen ($10\%$) is een ramp, terwijl 100 opzeggingen bij een label van 100.000 polissen ($0,1\%$) een topprestatie is.
    - Vergelijkbaarheid: Door deze ratio te berekenen, kan XGBoost patronen herkennen die voor álle labels gelden, ongeacht hun omvang. Het model leert zo welke economische factoren de relatieve uitstroom beïnvloeden.

2. Market Capture Index (Marktaandeel-proxy)
- De Logica: Deze index deelt jouw NewBusiness door de totale landelijke autoverkopen (CBS_Autoverkopen_Totaal).
- Waarom voor XGBoost: * Benchmarken: Als je NewBusiness met $5\%$ daalt, lijkt dat negatief. Maar als de totale automarkt met $20\%$ is ingestort, presteer je relatief gezien juist erg sterk.
    - Signal-to-Noise: Deze feature helpt het model om "marktruis" (algemene economische trends in de autosector) te scheiden van de effectiviteit van jouw eigen prijsbeleid of marketing. Het vertelt het model of de instroom komt door een groeiende markt of door jouw eigen concurrentiekracht.
    
3. Label Historical Avg Royementen (Baseline Encoding)
- De Logica: Dit berekent het historisch gemiddelde aantal royementen voor elk specifiek label. Dit is een vorm van Target Encoding.
- Waarom voor XGBoost: * DNA van het Label: Elk label heeft een eigen profiel. Een budgetlabel heeft vaak een structureel hogere churn dan een premium label.
    - Referentiekader: XGBoost ziet nu niet alleen het getal "80 royementen", maar kan dit vergelijken met het gemiddelde: "Dit label heeft normaal 100 royementen, dus 80 is deze maand een verbetering".
    - Snelheid van leren: Het model hoeft niet honderden bomen te bouwen om per label te ontdekken wat "normaal" is; je geeft die informatie direct mee als startpunt
    

Samenvatting: De kracht van de combinatie

Door deze drie features te combineren met de eerdere premie- en CPI-features, creëer je een model dat op drie niveaus tegelijk kijkt:
- Macro: Hoe doet de Nederlandse automarkt het? (Market_Capture_Index)
- Micro: Hoe reageert de klant op mijn prijs? (Retention_Rate)
- Identiteit: Wat is het normale gedrag van dit specifieke label? (Label_Historical_Avg_Royementen)

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2740 entries, 0 to 2739
Data columns (total 89 columns):
 #   Column                                                  Non-Null Count  Dtype         
---  ------                                                  --------------  -----         
 0   Jaar_Maand                                              2740 non-null   int64         
 1   Product                                                 2740 non-null   str           
 2   Label_Naam                                              2740 non-null   str           
 3   Beginstand                                              2740 non-null   int64         
 4   Jaarpremie_Beginstand                                   2740 non-null   float64       
 5   Eindstand                                               2740 non-null   int64         
 6   Jaarpremie_Eindstand                                    2740 non-null   float64       
 7   NewBusiness                                             2740 non-null  

In [31]:
# Oneindige waarden door delingen door 0 opruimen
df = df.replace([np.inf, -np.inf], np.nan)

In [32]:
df.to_csv('../../data/processed/regression/export_insurance_portfolio_data_forecasting_2015_2025_feature_engineering.csv', index=False)

print(f"Klaar! Dataset bevat nu {len(df.columns)} kolommen.")
print("Belangrijk: De eerste rijen van elk label bevatten NaN-waarden door de lag-features.")

Klaar! Dataset bevat nu 89 kolommen.
Belangrijk: De eerste rijen van elk label bevatten NaN-waarden door de lag-features.
